In [2]:
!pip install geemap
!pip install rioxarray
!pip install geopandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 68.2 MB/s eta 0:00:00
  Attempting uninstall: xarray
    Found existing installation: xarray 2025.12.0
    Uninstalling xarray-2025.12.0:
      Successfully uninstalled xarray-2025.12.0


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
^C


In [1]:
import ee
import geopandas as gpd
import geemap
from shapely.validation import make_valid
from shapely.ops import transform, unary_union
import os
import rioxarray as rxr
import pandas as pd
import calendar
from datetime import datetime, timedelta

# ==========================================
# 0. PENGATURAN DIREKTORI DINAMIS
# ==========================================
# Melacak lokasi folder tempat script ini dijalankan
BASE_DIR = os.getcwd()

# Menentukan lokasi GeoJSON (Asumsi berada di folder yang sama dengan script)
file_geojson = os.path.join(BASE_DIR, "33.05_kecamatan.geojson")

# Menentukan folder utama untuk output GSMaP
FOLDER_BASE_OUTPUT = os.path.join(BASE_DIR, "data", "gsmap")

tahun = 2026
bulan = 1

# 1. Inisialisasi Earth Engine
ee.Initialize(project='sismonbanjir')

# 2. Persiapan Batas Wilayah
if not os.path.exists(file_geojson):
    raise FileNotFoundError(f"File GeoJSON tidak ditemukan di: {file_geojson}. Pastikan file ada di folder yang sama dengan script.")

gdf = gpd.read_file(file_geojson)

if gdf.crs != "EPSG:4326":
    gdf = gdf.to_crs("EPSG:4326")

# ==========================================
# --- KUNCI PERBAIKAN (CLEANING TOPOLOGI) ---
# ==========================================
print("Membersihkan geometri GeoJSON yang cacat...")
gdf = gdf[gdf.geometry.notna()].copy()

def _to_2d(geom):
    if geom is None or geom.is_empty: return None
    return transform(lambda x, y, z=None: (x, y), geom)

def _extract_polygonal(geom):
    if geom is None or geom.is_empty: return None
    if geom.geom_type in ("Polygon", "MultiPolygon"): return geom
    if geom.geom_type == "GeometryCollection":
        polys = [g for g in geom.geoms if g.geom_type in ("Polygon", "MultiPolygon")]
        if not polys: return None
        return unary_union(polys)
    return None

def _clean_geom(geom):
    if geom is None or geom.is_empty: return None
    geom = _to_2d(geom)
    geom = make_valid(geom)
    geom = _extract_polygonal(geom)
    if geom is None or geom.is_empty: return None
    geom = geom.buffer(0)
    if geom is None or geom.is_empty: return None
    if not geom.is_valid:
        geom = make_valid(geom)
        geom = _extract_polygonal(geom)
    if geom is None or geom.is_empty or not geom.is_valid: return None
    return geom

gdf["geometry"] = gdf["geometry"].apply(_clean_geom)
gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
gdf = gdf[gdf.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
gdf.reset_index(drop=True, inplace=True)

if gdf.empty:
    raise ValueError("Semua geometri tidak valid setelah proses cleaning. Cek file GeoJSON sumber.")

print("Mengonversi ke Earth Engine...")
geojson_fc = gdf.__geo_interface__
batas_kebumen = ee.FeatureCollection(geojson_fc["features"])
# ==========================================

# ==========================================
# 3. FUNGSI PERULANGAN UNDUH PER JAM (GSMaP)
# ==========================================
def unduh_gsmap_perjam(tahun, bulan, batas_ee, output_dir):
    """
    Fungsi untuk mengunduh GSMaP per jam selama satu bulan penuh.
    Total file yang dihasilkan: ~744 file per bulan (24 jam x 31 hari)
    """
    os.makedirs(output_dir, exist_ok=True)

    # Mencari tahu ada berapa hari dalam bulan tersebut
    _, jml_hari = calendar.monthrange(tahun, bulan)

    print(f"\n--- Memulai Unduhan GSMaP Bulan {bulan:02d}-{tahun} ({jml_hari} Hari x 24 Jam) ---")
    print(f"Output folder: {output_dir}")

    for hari in range(1, jml_hari + 1):
        for jam in range(24): # LOOPING BARU: 24 Jam
            # Membentuk objek waktu per jam
            tgl_mulai_obj = datetime(tahun, bulan, hari, jam)
            # Batas akhir adalah 1 jam setelahnya
            tgl_akhir_obj = tgl_mulai_obj + timedelta(hours=1)

            # Format string ISO 8601 yang disukai Earth Engine (YYYY-MM-DDTHH:MM:SS)
            tgl_mulai = tgl_mulai_obj.strftime("%Y-%m-%dT%H:%M:%S")
            tgl_akhir = tgl_akhir_obj.strftime("%Y-%m-%dT%H:%M:%S")

            # Penamaan file ditambah atribut jam (contoh: gsmap_01_01_2026_1400.nc)
            file_name_base = f"gsmap_{hari:02d}_{bulan:02d}_{tahun}_{jam:02d}00"
            tif_path = os.path.join(output_dir, f"{file_name_base}.tif")
            nc_path = os.path.join(output_dir, f"{file_name_base}.nc")

            # Jika file NetCDF sudah ada, lewati
            if os.path.exists(nc_path):
                print(f"[{tgl_mulai}] Sudah ada, dilewati...")
                continue

            print(f"[{tgl_mulai}] Mengunduh...", end=" ")

            try:
                # 1. Tarik Data GSMaP V8 Operational
                dataset = ee.ImageCollection('JAXA/GPM_L3/GSMaP/v8/operational') \
                            .filter(ee.Filter.date(tgl_mulai, tgl_akhir))

                # Variabel GSMaP adalah 'hourlyPrecipRate'
                precipitation = dataset.select('hourlyPrecipRate').first().clip(batas_ee)

                # 2. Ekspor ke GeoTIFF
                geemap.ee_export_image(
                    precipitation,
                    filename=tif_path,
                    region=batas_ee.geometry(),
                    scale=11132, # KUNCI: Resolusi asli GSMaP adalah ~11.132 meter
                    file_per_band=False
                )

                # 3. Konversi GeoTIFF ke NetCDF
                with rxr.open_rasterio(tif_path, masked=True) as da_asli:
                    da = da_asli.squeeze("band", drop=True) if "band" in da_asli.dims else da_asli
                    da.name = "precipitation"

                    # Suntikkan metadata waktu (lengkap dengan jamnya)
                    waktu = pd.to_datetime(tgl_mulai_obj)
                    da = da.expand_dims(time=[waktu])

                    da.to_netcdf(nc_path)

                # 4. Hapus file TIF
                os.remove(tif_path)

                print("✓")

            except Exception as e:
                print(f"❌ (Error: {e})")

# ==========================================
# 4. EKSEKUSI FUNGSI
# ==========================================

# Membuat subfolder spesifik untuk bulan dan tahun (misal: data/gsmap/2026_01)
folder_output = os.path.join(FOLDER_BASE_OUTPUT, f"{tahun}_{bulan:02d}")

# Eksekusi unduhan
unduh_gsmap_perjam(tahun=tahun, bulan=bulan, batas_ee=batas_kebumen, output_dir=folder_output)

EEException: Please authorize access to your Earth Engine account by running

earthengine authenticate

in your command line, or ee.Authenticate() in Python, and then retry.